# 4.01 - Flight Feature Analysis
    In this notebook I do the following:

## 1.0 Compare summary statistics 

I compare the summary statistics of haunted places that were flagged as {"Plane_Crash", "Flying_Object", "Electronic_Malfunction"}. 

I hypothesize that these haunted places have elevated flight and aerodrome intersections. This is because:
- Electronic disturbances could be caused by nearby flights and landings
- Plane crashses could occur during landing, takeoff, or mid-flight
- Planes, helicopters, and balloons could be mistaken as UAPs in the night.


## 2.0 Compute Airport and Flight Summary Statistics

I compute the following summary statistics:
- Most haunted airports | Airports with the highest number of nearby haunted places.
    - best not go near these airports.
- Most haunted flights | flights that intersect with the most haunted places
    - you would not want to go on these flights.

## 3.0 Investigate Jaccard, Edit, and Cosine Clusters

### 1. Clustering Flying Events
- Clustering was performed only on haunted places that were flagged as {"Plane_Crash", "Flying_Object", "Electronic_Malfunction"}. 
- You can replicate results by pasting the *"indicies_of_interest"* variable into a .txt file and running **./dsci_550_a1/clusteringWorkflow.py**.
- All features were used in clustering

- **Jaccard**
    - Only yielded 2 clusters, not very informative
- **Cosine**
    - Only yielded 2 clusters. **This is likely because many of our added features are categorical and boolean**. Their direction in an embedding space does not capture characteristics of the data.
- **Edit**
    - Produced the most clusters (8) and the smallest cluster groups. 
    - Best cluster by Edit-distance: [3961, 6033, 10408, 7137, 926, 8722, 4111, 9408, 3739, 61]
        - 2 plane crashes and 8 flying objects all within 1 km of a christian church. 
        - **"It's a bird it's a plane it's an ... angel?"**

### 2. Clustering by Flying Features

I wanted to see if the added flying features could be used to cluster airborne related events together. 

You can replicate results by: 
- generating 1000 random indicies in range [0, 10922] and putting them in a .txt.
- running ***./dsci_550_a1/clusteringWorkflow.py** 
- passing in columns **[ "Aerodrome_Proximity", "Aerodrome_Count", "Flight_Intersection_Count", "Flight_HighTraffic"]** to cluster by.

I'll save you the trouble though, results were not encouraging. 

- **cosine** performed the best, but was only able to get a cluster with 3% flying objects. **This is high compared to the 1% proportion accross our dataset**.
- **# of flight intersections could be an informative feature**. This cluster had an average of 28 flight intersections while the sample had an average of 20 flight intersection. 
- Through interesting, these result are still is not encouraging.

### Conclusion
- Added air-related features cannot be used to cluster air-related events
- Edit-distance provided the most interesting and specific clusters of all methods


## Load Data 

In [665]:
# System Path #
import os
import sys 

parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

# Pandas, json and runtime #
import pandas as pd
from tqdm import tqdm
import json
import time
tqdm.pandas()

# Tika #
import tika as tk
from tika import parser

# Iterators #
from collections import Counter
from itertools import chain
import re

# local functions #
from dsci_550_a1.parsingFunctions import extractSequences
from dsci_550_a1.unpack_circles import unpack_cluster


## Features Added df_haunted_places ##
outfile = "../data/processed/haunted_places_features_added.tab"

df_haunted_places = pd.read_csv(outfile, sep = "\t")


## Filter Indicies that contain flights ##

Flying_Object_idx = df_haunted_places[df_haunted_places["Event_Type"].str.contains('Flying_Object')].index.tolist()
Electronic_Malfunction_idx = df_haunted_places[df_haunted_places["Event_Type"].str.contains('Electronic_Malfunction')].index.tolist()
Plane_Crash_idx = df_haunted_places[df_haunted_places["Event_Type"].str.contains('Plane_Crash')].index.tolist()
 
indicies_of_interest = list(chain(Flying_Object_idx + Electronic_Malfunction_idx + Plane_Crash_idx))


## 1. Summary statistic comparison

In [666]:
## How many haunted places are in high traffic areas? ##

print("Accross our dataset we have:")

for column in ['Aerodrome_Proximity', 'Flight_HighTraffic']:
    ratio = df_haunted_places[column].value_counts('True')[True]
    print(f"{ratio:3g}% haunted places flagged True for '{column}'", sep = "\n")
    


Accross our dataset we have:
0.76683% haunted places flagged True for 'Aerodrome_Proximity'
0.310135% haunted places flagged True for 'Flight_HighTraffic'


In [667]:
df_haunted_places['Flight_HighTraffic'].value_counts('True')

Flight_HighTraffic
False    0.689865
True     0.310135
Name: proportion, dtype: float64

In [668]:
## Summary statistics for flight-related entries
column_filter = [ 'Aerodrome_Proximity', 'Aerodrome_Count', 'Flight_Intersection_Count', 'Flight_HighTraffic']
df_filtered = df_haunted_places.loc[indicies_of_interest, :]
print(df_filtered[column_filter].describe()) 

       Aerodrome_Count  Flight_Intersection_Count
count       199.000000                 199.000000
mean          1.467337                  16.296482
std           1.496693                  46.911345
min           0.000000                   0.000000
25%           0.000000                   0.000000
50%           1.000000                   4.000000
75%           2.000000                  10.500000
max           9.000000                 346.000000


In [669]:
## Summary Statistics for non-flight-related entries
print(df_haunted_places[column_filter].describe())

       Aerodrome_Count  Flight_Intersection_Count
count     10992.000000               10992.000000
mean          1.836699                  21.642103
std           3.517117                  67.570107
min           0.000000                   0.000000
25%           1.000000                   0.000000
50%           1.000000                   6.000000
75%           2.000000                  14.000000
max          98.000000                1504.000000


In [670]:
## Compute difference
df_filtered[column_filter].describe() - df_haunted_places[column_filter].describe() 

,Aerodrome_Count,Flight_Intersection_Count
count,-10793.000000,-10793.000000
mean,-0.369363,-5.345621
std,-2.020424,-20.658762
min,0.000000,0.000000
25%,-1.000000,0.000000
50%,0.000000,-2.000000
75%,0.000000,-3.500000
max,-89.000000,-1158.000000


In [671]:
print("Haunted places flagged as plane crashes, floating objects, or electronic malfunction had on average:")
for column in column_filter:
    try: 
        diff = df_filtered[column].describe().loc['mean'] - df_haunted_places[column].describe().loc['mean']
    except KeyError:
        diff = df_filtered[column].value_counts('True').loc[True] - df_haunted_places[column].value_counts('True').loc[True] 
    if diff < 0:
        diff = f"{abs(diff):3g} lower"
    else:
        diff = f"{abs(diff):3g} higher"
    print(f"{diff} {column}", sep = "\n"
    )


Haunted places flagged as plane crashes, floating objects, or electronic malfunction had on average:
0.0582877 lower Aerodrome_Proximity
0.369363 lower Aerodrome_Count
5.34562 lower Flight_Intersection_Count
0.0588784 lower Flight_HighTraffic


### **Results**

- Haunted places with plane crashes actually have a lower than average *Aerodrome_count* and *Flight_Intersection_Count*. 
- However, they also have a significantly lower standard deviation. **This is likely because these haunted places are far from major cities**. They do not have large regional airports near them leading to low flight intersection counts. 



## 2. Most Haunted Flights and Airports

- For this analysis I wanted to see what the haunted place dataset could tell us about the [openFlights](https://raw.githubusercontent.com/jpatokal/openflights/master/data/routes.dat) and [ourAirports](https://ourairports.com/data/) datasets. 

- Mainly I wanted to see what flights I should avoid in the future.


In [672]:
## Flight Intersection Data
with open("../data/processed/flight_proximity_data.json") as f:
    flight_intersection_data = json.load(f)
    
## Airport Intersection Data
with open("../data/processed/airport_proximity_data.json") as f:
    airport_intersection_data = json.load(f)


## Counting most common Airports ##
airport_counter = Counter()

for i, data in airport_intersection_data.items():
    for airport in data["Airports"]:
        airport_counter[airport["Name"]] += 1

## Counting most common Flights ##
flight_counter = Counter()
for i, data in flight_intersection_data.items():
    
    for flight in data["Routes"]:
        source_airport, dest_airport = flight["Source_Airport"], flight["Dest_Airport"]
        flight_name = f"{source_airport} -> {dest_airport}"
        flight_counter[flight_name] += 1

## Fun Stats ##

# Haunted places with most airports #
most_airports = df_haunted_places.sort_values(by = "Aerodrome_Count", ascending = False).head(10).index.tolist()
# Airports with the most haunted places #
most_haunted_airports = "\n\t".join(f"{key} : {value}" for key, value in airport_counter.most_common(10))
# Flights with the most haunted places #
most_haunted_flights = "\n\t".join(f"{key} : {value}" for key, value in flight_counter.most_common(10))
# Haunted places with most flights #
most_flights = df_haunted_places.sort_values(by = "Flight_Intersection_Count", ascending = False).head(10).index.tolist()

In [673]:
## Show haunted places that have the most flights intersecting them.
df_haunted_places.loc[most_flights]

,City,Country,Description,Location,State,State_Abbrev,Longitude,Latitude,City_Longitude,City_Latitude,...,Time_of_Day,Total_Deaths,Percent_Under_21,Average_Mental_Health_Days,Average_Poor_Health_Days,Depression_Prevalence,Haunted_Place_Proximity,Distance_to_Nearest_Worship,Religion_Intersection,Daylight_Duration_Hours
8286,Forest Park,United States,it has a shed and when you walk towards the sh...,cemetery across the street from the teen center,Georgia,GA,-84.369092,33.622054,-84.369092,33.622054,...,Evening,"3,953",3.70%,5.51,6.18,0.272,True,1392.39,christian,9.907
8202,Atlanta,United States,Years ago a rock singer was killed on stage by...,Hi-Fi Buys Amphitheater,Georgia,GA,-84.397164,33.703548,-84.387982,33.748995,...,Unknown,"3,953",3.70%,5.51,6.18,0.272,True,848.30,christian,10.005
6851,Des Plaines,United States,Back in the 1930's when the Academy was used a...,Maryville Academy,Illinois,IL,-87.897529,42.066829,-87.883399,42.033362,...,Unknown,"2,960",3.40%,4.07,5.49,0.157,True,837.21,christian,9.652
6889,Elmhurst,United States,At one time it was also called Spaso Speakeasy...,The old Stone Cottage Pizzeria,Illinois,IL,-87.940342,41.899474,-87.940342,41.899474,...,Unknown,"2,960",3.40%,4.07,5.49,0.157,True,801.41,christian,9.252
7236,Rosemont,United States,The hotel is an atrium style hotel with 11 flo...,Sheraton Gateway Suites Hotel,Illinois,IL,-87.883407,41.997269,-87.872160,41.986751,...,Evening,"2,960",3.40%,4.30,5.71,0.174,True,2030.72,christian,9.246
6850,Des Plaines,United States,"Late at night near the juice bar of the club, ...",Galaxy Club,Illinois,IL,-87.883399,42.033362,-87.883399,42.033362,...,Evening,"2,960",3.40%,4.07,5.49,0.157,True,396.00,christian,9.225
6749,Chicago,United States,dead passengers and flight attendants pilots. ...,O'Hare airport,Illinois,IL,-87.907321,41.974162,-87.629798,41.878114,...,Unknown,"2,960",3.40%,4.95,5.85,0.193,True,4509.96,christian,9.243
7118,Mt. Prospect,United States,The shadow of a very tall man makes occasional...,Mrs. P &amp; Me Restaurant,Illinois,IL,-87.935181,42.062150,-87.937291,42.066417,...,Unknown,"5,151",3.50%,3.75,4.75,0.163,True,424.19,christian,9.443
6888,Elmhurst,United States,reports of an apparition of a man with a lante...,Single track train track,Illinois,IL,-87.961677,41.929122,-87.940342,41.899474,...,Unknown,"2,960",3.40%,4.07,5.49,0.157,True,1320.10,christian,11.809
6783,Chicago,United States,Reports have been made that you can see black ...,Robinson Woods,Illinois,IL,-87.853299,41.965043,-87.629798,41.878114,...,Unknown,"5,151",3.50%,4.95,5.85,0.193,True,1728.60,christian,9.267


In [674]:
df_haunted_places.loc[most_airports]

,City,Country,Description,Location,State,State_Abbrev,Longitude,Latitude,City_Longitude,City_Latitude,...,Time_of_Day,Total_Deaths,Percent_Under_21,Average_Mental_Health_Days,Average_Poor_Health_Days,Depression_Prevalence,Haunted_Place_Proximity,Distance_to_Nearest_Worship,Religion_Intersection,Daylight_Duration_Hours
2811,Los Angeles,United States,Cabin The First Interstate Bldg. (now called t...,Downtown,California,CA,-118.246769,34.040713,-118.243685,34.052234,...,Unknown,"15,443",2.50%,4.11,5.37,0.185,True,984.16,christian,9.979
2824,Los Angeles,United States,Walnut - City Of Indusry - St. Mary's Catholic...,LA County,California,CA,-118.243685,34.052234,-118.243685,34.052234,...,Evening,"15,443",2.50%,4.11,5.37,0.185,True,662.97,christian,9.981
2845,Los Angeles,United States,Reports of a residual scene of a murder happen...,The over pass over Pasadena Free way,California,CA,-118.243685,34.052234,-118.243685,34.052234,...,Unknown,"15,443",2.50%,5.37,6.36,0.254,True,662.97,christian,9.975
2852,Los Angeles,United States,The vaudeville-era Palace at 6th and Broadway ...,Palace Theatre,California,CA,-118.252351,34.045726,-118.243685,34.052234,...,Unknown,"15,443",2.50%,5.37,6.36,0.254,True,174.48,christian,10.095
2829,Los Angeles,United States,"Security Officers report of Haunting on the 2,...",Los Angeles City Hall,California,CA,-118.242653,34.053714,-118.243685,34.052234,...,Evening,"15,443",2.50%,4.11,5.37,0.185,True,641.00,christian,9.970
2800,Los Angeles,United States,"The spirit of a young girl by the name of ""Ara...",Belmont High School,California,CA,-118.262934,34.061457,-118.243685,34.052234,...,Unknown,"15,443",2.50%,4.50,6.06,0.238,True,181.69,christian,9.970
2840,Los Angeles,United States,There have been reports of a woman with red he...,Our Lady of Loretto Elementary,California,CA,-118.264006,34.066953,-118.243685,34.052234,...,Unknown,"15,443",2.50%,4.11,5.37,0.185,True,594.11,christian,9.991
2814,Los Angeles,United States,Sometimes the elevator will stop on certain fl...,Figueroa hotel,California,CA,-118.263925,34.045484,-118.243685,34.052234,...,Evening,"15,443",2.50%,4.11,5.37,0.185,True,407.08,christian,9.976
2797,Los Angeles,United States,"Formerly Pabst Blue Ribbon, this complex house...",Angel City Brewery,California,CA,-118.237693,34.046261,-118.243685,34.052234,...,Unknown,"15,443",2.50%,4.50,6.06,0.238,True,492.44,christian,9.978
2807,Los Angeles,United States,Apartment on Centennial St - Footsteps in the ...,Chinatown,California,CA,-118.238331,34.062334,-118.243685,34.052234,...,Unknown,"15,443",2.50%,4.50,6.06,0.238,True,203.10,christian,9.980


In [675]:
## Aerodrome Proximity Printout ##
print("-" * 50, "Aerodrome Proximity", "-" * 50)
print(f"Top 10 most haunted airports:", most_haunted_airports, sep = "\n\t")
print(f"Top 10 haunted places with most airports:", most_airports, sep = "\n\t")
print("-" * 100, end = "\n\n")

## Flight Proximity Printout ##
print("-" * 50, "Flight Proximity", "-" * 50, )
print(f"Top 10 most haunted flights:", most_haunted_flights, sep = "\n\t")
print(f"Top 10 haunted places with most flights:", most_flights, sep = "\n\t")
print("-" * 100, end = "\n\n")


-------------------------------------------------- Aerodrome Proximity --------------------------------------------------
Top 10 most haunted airports:
	Los Angeles International Airport : 329
	John Wayne Orange County International Airport : 311
	Ontario International Airport : 266
	Logan International Airport : 216
	Detroit Metropolitan Wayne County Airport : 205
	Newark Liberty International Airport : 191
	Chicago O'Hare International Airport : 168
	Philadelphia International Airport : 166
	LaGuardia Airport : 161
	San Francisco Bay Oakland International Airport : 160
Top 10 haunted places with most airports:
	[2811, 2824, 2845, 2852, 2829, 2800, 2840, 2814, 2797, 2807]
----------------------------------------------------------------------------------------------------

-------------------------------------------------- Flight Proximity --------------------------------------------------
Top 10 most haunted flights:
	LAS -> LAX : 738
	LAX -> LAS : 738
	ATL -> BOS : 648
	BOS -> ATL : 

In [676]:
df_american_airports = pd.read_csv("../data/joined_datasets/american_airports.tsv", sep = "\t")


for type in pd.unique(df_american_airports['Type']).tolist():
    x = 0
    airport_names_with_type = df_american_airports.loc[df_american_airports['Type'] == f'{type}', 'Name'].tolist()
    filtered_counter = Counter()
    ## Counting most common Airports ##
    filtered_counter = Counter()
    for key, val in airport_counter.most_common():
        if key in airport_names_with_type:
            filtered_counter[key] = val 
            x+= 1
        if x == 10:
            break

    print("-" * 50, f"{type}", "-" * 50)
    print(f"Top 10 most haunted {type}(s):", "\n\t".join(f"{key} : {value}" for key, value in filtered_counter.most_common(10)), sep = "\n\t")

print("-" * 100, end = "\n\n")

-------------------------------------------------- heliport --------------------------------------------------
Top 10 most haunted heliport(s):
	Mercy Hospital Heliport : 27
	Baptist Medical Center Heliport : 27
	Del Rio Heliport : 24
	Chevron Place Heliport : 23
	New Orleans Downtown Heliport : 23
	University Medical Center New Orleans Heliport : 23
	Tulane Medical Center Heliport : 23
	Methodist Hospital Metropolitan Helipad : 23
	World Trade Center Heliport : 22
	600 Grant Street Rooftop Heliport : 21
-------------------------------------------------- small_airport --------------------------------------------------
Top 10 most haunted small_airport(s):
	Flabob Airport : 19
	Gettysburg Regional Airport : 18
	Bowman Field : 16
	Downtown Airport : 16
	Mackinac Island Airport : 12
	San Gabriel Valley Airport : 12
	Brackett Field : 12
	Reid-Hillview Airport of Santa Clara County : 12
	Columbus Municipal Airport : 12
	Williamsburg Jamestown Airport : 12
-----------------------------------

### **Results**
**Haunted Places  with Most Flights**
- Nothing super interesting here. They are just in Chicago or near Atlanta

**Haunted Places with Most Airports**
- Again nothing interesting. Just haunted places in Los Angeles.

**Most Haunted Flights**
- You really do not want to fly from Los angeles to Las Vegas. Who would've thought


**Most Haunted Airports**
- LAX and John Wayne are the most haunted airports. This is likely because LA has a disproportionately high amount of haunted places. 
- However, when we sort by airport type we get more interesting information.
    - **You do not want to fly a helicopter in New orleans** [New Orleans Downtown Heliport, University Medical Center New Orleans Heliport, Tulane Medical Center Heliport]
    - **The Disney balloonport is bad news**
    - **Beware of the Alamo**The San Antonio airport is the spookiest medium sized airport
    - You can run notebook 3.03 to visualize these airports :\)





#### Saving Counts for Plotting in 3.03 Notebook

Saving the counter objects as json so we can use them as input in Plotly

In [677]:
df_american_airports = pd.read_csv("../data/joined_datasets/american_airports.tsv", sep = "\t")
airport_counter = Counter()

for i, data in airport_intersection_data.items():
    for airport in data["Airports"]:
        airport_counter[airport["Name"]] += 1

airport_counts_by_type = {}
for type in pd.unique(df_american_airports['Type']).tolist():

    airport_names_with_type = df_american_airports.loc[df_american_airports['Type'] == f'{type}', 'Name'].tolist()
    filtered_counter = Counter()

    for key, val in airport_counter.most_common():
        if key in airport_names_with_type:
            filtered_counter[key] = val 


    airport_counts_by_type[type] = filtered_counter.most_common()

with open('../data/processed/airport_haunted_place_counts_names.json', 'w') as f:
    json.dump(dict(airport_counts_by_type), f, indent=4)


## Counting most common Flights ##
flight_counter = Counter()
for i, data in flight_intersection_data.items():
    
    for flight in data["Routes"]:
        source_airport, dest_airport = flight["Source_Airport"], flight["Dest_Airport"]
        flight_name = f"{source_airport} -> {dest_airport}"
        flight_counter[flight_name] += 1

with open('../data/processed/flight_haunted_place_counts.json', 'w') as f:
    json.dump(dict(flight_counter.most_common()), f, indent=4)

## 3.0 Investigate Jaccard, Edit, and Cosine Clusters


### Clustering Air Related Events

What can our other features tell us about flight related events?

In [685]:
# jaccard_clusters = unpack_cluster('../clustering/Flight_Features/flightsAllFeatures/jaccard/visualization/clusters.json')
# cosine_clusters = unpack_cluster('../clustering/Flight_Features/flightsAllFeatures/cosine/visualization/clusters.json')
edit_clusters = unpack_cluster('../clustering/Flight_Features/flightsAllFeatures/edit-distance/visualization/clusters.json')


column_order = [
    "Description", 'Location', 'City', 'State_Abbrev', 'Longitude', 'Latitude',
    'Haunted_Places_Date', 'Time_of_Day',
    # Required Features
    'Audio_Evidence', 'Visual_Evidence', 'Event_Type', 'Apparition_Type', 'Haunted_Places_Witness_Count', 'Haunted_Place_Proximity', 
    # Alcohol
    'Total_Deaths', 'Percent_Under_21', 
    #Airport
    'Aerodrome_Proximity', 'Aerodrome_Count', 'Flight_Intersection_Count', 'Flight_HighTraffic', 
    # Religion
    'Religion_Intersection', 'Distance_to_Nearest_Worship', 
    # Mental Health
    'Average_Poor_Health_Days', 'Depression_Prevalence', 'Average_Mental_Health_Days',
    # Daylight
    'Daylight_Duration_Hours'
    
]

cluster_names = edit_clusters.keys()

print(cluster_names)

cluster_name = 'cluster 8'

df_slice = df_haunted_places.loc[edit_clusters[f"{cluster_name}"]].drop(columns = ["Country", "State", "City_Longitude", "City_Latitude"])[column_order]
df_slice


dict_keys(['cluster 0', 'cluster 1', 'cluster 2', 'cluster 3', 'cluster 4', 'cluster 5', 'cluster 6', 'cluster 7', 'cluster 8'])


,Description,Location,City,State_Abbrev,Longitude,Latitude,Haunted_Places_Date,Time_of_Day,Audio_Evidence,Visual_Evidence,...,Aerodrome_Proximity,Aerodrome_Count,Flight_Intersection_Count,Flight_HighTraffic,Religion_Intersection,Distance_to_Nearest_Worship,Average_Poor_Health_Days,Depression_Prevalence,Average_Mental_Health_Days,Daylight_Duration_Hours
3961,Little Vietnamese boy who walks the museum at ...,Air Force Museum,Dayton,OH,-84.109382,39.780796,2025-01-01,Evening,False,True,...,True,3,12,True,christian,3003.93,5.85,0.183,3.77,9.461
6033,there have been multiple sightings by locals. ...,streets around Eagle Ave. &amp; the Train Tracks,West Hempstead,NY,-73.908288,40.823023,2025-01-01,Unknown,True,True,...,True,3,311,True,NaN,146.27,5.40,0.227,4.61,9.373
10408,On the corner of French and Main street in Wat...,Main Street Graveyard,Watertown,CT,-73.123343,41.607830,2025-01-01,Evening,True,True,...,False,0,13,True,christian,2826.26,NaN,NaN,NaN,9.281
7137,The legend is so old that nobody really knows ...,Green Lantern Road,Olney,IL,-88.085315,38.730881,2025-01-01,Evening,False,True,...,True,1,2,False,christian,302.47,4.75,0.163,3.75,9.571
926,it is known that outside going towards Bakers ...,Bakers Crossroads,Patton,PA,-78.650301,40.633956,2025-01-01,Evening,False,True,...,False,0,0,False,christian,104.70,5.83,0.192,4.63,9.380
8722,This place contains many occurrences that cann...,Rocky Boy Indian Reservation,Rocky Boy,MT,-109.800758,48.250265,2025-01-01,Evening,False,True,...,False,0,0,False,christian,2069.55,5.83,0.226,4.74,11.154
4111,"In the 50s or 60s a small plane with a man, hi...",Airplane Hollow,Nelsonville,OH,-82.231816,39.458681,1960-01-01,Unknown,True,False,...,False,0,4,False,christian,305.41,5.60,0.229,4.50,11.260
9408,the big South Tunnel train tracks - Believed t...,Portland,Sumner county,TN,-86.516383,36.581709,2025-01-01,Unknown,False,True,...,True,2,0,False,christian,76.69,4.90,0.224,4.01,9.773
3739,It’s said that if you drive by the cemetery la...,Bethlehem Methodist Church,Munford,AL,-85.957325,33.500934,2025-01-01,Evening,True,True,...,False,0,8,False,christian,158.27,4.72,0.139,4.05,10.026
61,An old farmer went insane and killed his famil...,Hikyes Tomb,Cheboygan,MI,-84.474480,45.646956,2025-01-01,Evening,True,False,...,True,1,0,False,NaN,156.50,5.72,0.234,4.56,8.841


#### Results
- edit-distance performed the best producing 8 clusters
- Cluster 8 is particularly interesting
    - Every entry is on average 1 km away from a christian institution
    - **it's a bird it's a plane it's an ... Angel**


### Clustering Using Air-Related Features

What can our added flight features tell us about our data?

In [679]:
## Flag relevant values and specify column
col = 'Event_Type'

## Splitting multi-event-type entries and geting unique events
flagged_values = pd.unique(df_haunted_places[f'{col}']).tolist()
unique_flags = []
for entry in flagged_values:
    entries = list(map(lambda x: x.strip(), entry.split('|')))
    [unique_flags.append(entry) for entry in entries if entry not in unique_flags]

#############################################
## Assign Flagged values in hierarchy
## Events that appear less often in the dataframe are at the top of our hierarchy


flag_ranks = {}
for flag in unique_flags:
    flag_ranks[flag] = 0
    for key in  df_haunted_places[col].value_counts().keys():
        if flag in key:
            flag_ranks[flag] += df_haunted_places[col].value_counts()[key]

flag_ranks = sorted(flag_ranks.keys(), key = lambda x: flag_ranks[x], reverse = False)


## Assigning each entry the rarest event type if it has more than one event-type

for flag in flag_ranks:
    for idx in df_haunted_places.index:
        if flag in df_haunted_places.loc[idx, col]:
            df_haunted_places.loc[idx,col] = flag
df_haunted_places['Event_Type'].value_counts()

Event_Type
Unknown                   3827
Supernatural              2771
Violence                  2675
Accident/Disaster         1536
Flying_Object              111
Electronic_Malfunction      56
Plane_Crash                 16
Name: count, dtype: int64

In [697]:
#############################################
## Load cluster output and indicies used in clustering

## Cluster Output
clusters = unpack_cluster('../clustering/Flight_Features/allFlightFeatures/edit-distance/visualization/clusters.json')

## Indicies used in clustering (not uploaded to github)
with open('../clustering/Flight_Features/allFlightFeatures/edit-distance/metadata.json', 'r') as f:
    # data = f.read()
    data = json.load(f)['number of files']
print(data)
## Turn indicies into list and take corresponding rows from haunted places data
data = data.replace('[', '')
data = data.replace(']', '')
idxs = data.split(',')

idxs = map(lambda x: int(x), idxs)
sampled_df = df_haunted_places.loc[idxs]

#############################################
## Find cluster with the highest proportion of values in column

# Initialize max and res to store cluster name
mx = 0 
x = 0
res = ''
col = 'Event_Type'
vals = ['Flying_Object', 'Electronic_Malfunction', 'Plane_Crash']

for _ in range(len(cluster_names)):
    cluster_name = f'cluster {x}'
    df_slice = df_haunted_places.loc[clusters[f"{cluster_name}"]].drop(columns = ["Country", "State", "City_Longitude", "City_Latitude"])[column_order].drop_duplicates()

    # if there is a higher proportion of entries with desired values in cluster, save the cluster. It becomes the new max
    if (df_slice[col].value_counts('True').reindex(vals, fill_value = 0).sum() > mx):
        mx = df_slice[col].value_counts('True').reindex(vals, fill_value = 0).sum()
        res = cluster_name

    x += 1

## Take rows corresponding to optimal cluster

df_cluster = df_haunted_places.loc[clusters[f"{res}"]].drop(columns = ["Country", "State", "City_Longitude", "City_Latitude"])[column_order].drop_duplicates()

## Check average values for each feature we clustered by and compare to the random sample 

cluster_cols = ['Aerodrome_Proximity', 'Aerodrome_Count', 'Flight_Intersection_Count', 'Flight_HighTraffic']

## Printout


for col in cluster_cols:
    print('-' * 50 + col + '-' * 50)
    try:
        print(col, f"Sample : {sampled_df[col].describe().loc['mean']}", f"{res} : {df_cluster[col].describe().loc['mean']}", sep = "\n")
    except KeyError:
        print(col, f"Sample : {sampled_df[col].value_counts()[True] / sampled_df[col].value_counts().sum()}", f"{res} : {df_cluster[col].value_counts()[True] / df_cluster[col].value_counts().sum()}", sep = "\n")
print('-' * 100+  '\n')

print('Proportion of Each Event Type in Sample: \n', sampled_df['Event_Type'].value_counts('True'), end = "\n" + "-" * 50 + "\n")
print(f'Proportion of Each Event Type in {res}: \n', df_cluster['Event_Type'].value_counts('True'), end = "\n" + "-" * 50)


[2660, 9986, 8158, 7771, 9037, 9692, 8955, 112, 7320, 8858, 6576, 1218, 4078, 1102, 10647, 795, 1194, 9524, 6595, 5697, 5956, 4197, 4069, 9023, 222, 4540, 5785, 1672, 5287, 3506, 5136, 1744, 8959, 8894, 7068, 10308, 7815, 7181, 3658, 3256, 9961, 947, 6592, 3392, 184, 6099, 8393, 1969, 478, 4563, 3936, 10303, 7159, 6252, 5031, 6265, 6562, 8246, 7171, 1895, 4462, 1324, 4332, 2518, 7981, 1249, 9894, 3634, 4051, 10974, 3999, 3266, 9994, 3971, 7140, 4100, 3768, 3587, 6701, 3860, 7500, 502, 2038, 9006, 3560, 7693, 2679, 9515, 4986, 9680, 7311, 1304, 9523, 3887, 386, 705, 941, 631, 4381, 10154, 2058, 4919, 5060, 171, 2391, 9478, 4467, 3004, 4702, 4409, 683, 2405, 5430, 3305, 207, 10714, 1615, 7975, 2136, 8692, 8738, 6319, 4052, 194, 10014, 6118, 3047, 6503, 775, 2339, 6836, 4959, 6297, 10845, 10508, 3204, 8012, 4426, 5872, 9339, 512, 2671, 3680, 1576, 2127, 4654, 3435, 1651, 10238, 3373, 5533, 5094, 6148, 6888, 4892, 902, 2250, 2002, 6500, 8136, 4750, 5803, 8787, 3839, 8944, 4049, 3750, 14, 1

### Conclusion
- Our best performing cluster was cluster 6 with **3% of entries having Flying Objects**.
- This is **triple the 1% average accross the dataset**.
- Entries in cluster 6 had on average 9 more flight intersections than the random sample
    - **flight intersection count could be an indicator of flying objects.**
- **Overal Conclusion: Flight features are not an effective way to cluster airborne related events**
